# Notebook 2 — Feature Engineering

Compute technical indicators and inspect the final feature matrix used for LSTM input.

In [ ]:
import sys
sys.path.insert(0, '..')

from src.data.fetcher import DataFetcher
from src.data.processor import DataProcessor
from src.utils.config import load_config
import plotly.graph_objects as go
from plotly.subplots import make_subplots

config = load_config()
config['data']['source'] = 'yfinance'
config['data']['timeframe'] = '1Day'

fetcher = DataFetcher(config)
processor = DataProcessor(config)

In [ ]:
raw_df = fetcher.fetch('AAPL', start='2020-01-01', end='2024-12-31')
feat_df = processor.process(raw_df)

print(f'Feature columns ({feat_df.shape[1]}):')  
print(feat_df.columns.tolist())
feat_df.head()

In [ ]:
# Visualize RSI and MACD alongside price
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    subplot_titles=['Close Price', 'RSI', 'MACD'],
                    vertical_spacing=0.05, row_heights=[0.5, 0.25, 0.25])

fig.add_trace(go.Scatter(x=feat_df.index, y=feat_df['close'], name='Close'), row=1, col=1)
fig.add_trace(go.Scatter(x=feat_df.index, y=feat_df['bb_upper'], name='BB Upper',
                         line=dict(dash='dash', color='gray')), row=1, col=1)
fig.add_trace(go.Scatter(x=feat_df.index, y=feat_df['bb_lower'], name='BB Lower',
                         line=dict(dash='dash', color='gray'), fill='tonexty',
                         fillcolor='rgba(200,200,200,0.15)'), row=1, col=1)

fig.add_trace(go.Scatter(x=feat_df.index, y=feat_df['rsi'], name='RSI', line=dict(color='purple')), row=2, col=1)
fig.add_hline(y=70, line_dash='dash', line_color='red', row=2, col=1)
fig.add_hline(y=30, line_dash='dash', line_color='green', row=2, col=1)

fig.add_trace(go.Scatter(x=feat_df.index, y=feat_df['macd'], name='MACD', line=dict(color='blue')), row=3, col=1)
fig.add_trace(go.Scatter(x=feat_df.index, y=feat_df['macd_signal'], name='Signal', line=dict(color='orange')), row=3, col=1)
fig.add_trace(go.Bar(x=feat_df.index, y=feat_df['macd_diff'], name='Histogram'), row=3, col=1)

fig.update_layout(title='AAPL — Technical Indicators', height=800, xaxis_rangeslider_visible=False)
fig.show()

In [ ]:
# Correlation heatmap of features
import plotly.figure_factory as ff
import numpy as np

corr = feat_df.corr().round(2)
fig = ff.create_annotated_heatmap(
    z=corr.values,
    x=list(corr.columns),
    y=list(corr.index),
    colorscale='RdBu',
    reversescale=True
)
fig.update_layout(title='Feature Correlation Matrix', height=700)
fig.show()

In [ ]:
# Build sequences and inspect shapes
X, y = processor.build_sequences(feat_df, target_col='close')
X_train, y_train, X_val, y_val, X_test, y_test = processor.train_val_test_split(X, y)

print(f'X shape:       {X.shape}  → (samples, seq_len={X.shape[1]}, features={X.shape[2]})')
print(f'X_train:       {X_train.shape}')
print(f'X_val:         {X_val.shape}')
print(f'X_test:        {X_test.shape}')